# Lab 2b — Churn Classifiers + the Overfitting Exhibit
**SDA-AIE-111 · Day 2 · Hour 2 · 50 minutes · pairs**

**Objective.** Beat the Lab 1 heuristic with logistic regression; produce the odds-ratio sentence for the CRM team; generate the course's key overfitting plot from your own runs; watch k-NN collapse without scaling.


In [ ]:
# --- Course setup (identical in every lab) -----------------------------------
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

def find_repo_root() -> Path:
    """Locate the course repo root by looking for data/manafeth_customers.parquet."""
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "data" / "manafeth_customers.parquet").exists():
            return cand
    raise FileNotFoundError(
        "Course data not found — copy the Manafeth data package files into <repo>/data/")

ROOT = find_repo_root()
DATA = ROOT / "data"
sys.path.insert(0, str(ROOT / "src"))
print("repo root:", ROOT)

In [ ]:
from sklearn.model_selection import train_test_split

customers = pd.read_parquet(DATA / "manafeth_customers.parquet")
LEAKY = ["refund_issued", "support_ticket_after_snapshot", "next_month_orders"]
X_all = customers.drop(columns=["churned_30d", "customer_id", *LEAKY])
y_all = customers["churned_30d"]
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.20, stratify=y_all, random_state=RANDOM_STATE)

num_cols = ["orders_per_month", "avg_basket_sar", "days_since_last_order",
            "tenure_months", "distinct_categories", "promo_usage_rate"]

# Working split for today (CV replaces this in Module 4)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train[num_cols], y_train, test_size=0.25, stratify=y_train,
    random_state=RANDOM_STATE)

def recall_at_budget(y_true, scores, budget=0.20):
    n = int(budget * len(y_true))
    top = np.argsort(-np.asarray(scores))[:n]
    return np.asarray(y_true)[top].sum() / np.asarray(y_true).sum()

## Task 1 — Logistic regression: the probability is the product *(15 min)*

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score

# TODO: scale features (fit the scaler on X_tr ONLY), fit
#       LogisticRegression(class_weight="balanced", max_iter=1000)
# TODO: report PR-AUC and recall@20% — rank by predict_proba[:, 1], never by predict!


## Task 2 — The odds-ratio sentence *(5 min)*

`exp(coef)` on standardised features = the odds multiplier per one standard deviation. Write the one sentence the CRM team would quote.


In [ ]:
# TODO: compute np.exp(clf.coef_[0]) and print odds ratios sorted by |log-odds|
# TODO: write the one-sentence CRM quote in the markdown cell below


## Task 3 — The overfitting exhibit: tree depth sweep *(15 min)*

Training score rises forever; validation tells the truth. This plot is reused all week — Module 7 generalises it into hyperparameter search.


In [ ]:
from sklearn.tree import DecisionTreeClassifier

# TODO: sweep max_depth 1..20; record train and validation PR-AUC per depth
# TODO: plot both curves, mark the best validation depth with a vertical line
# TODO: in one sentence — what happens after the peak, and why?


## Task 4 — k-NN and the scaling collapse *(10 min)*

Distance-based models live or die by feature scale: `tenure_months` (0–60) crushes `promo_usage_rate` (0–1) unless you scale.


In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# TODO: fit k-NN with k in {1, 5, 25, 101} on scaled AND unscaled features
# TODO: compare validation PR-AUC — explain the gap with a two-column distance
#       calculation (tenure_months range vs promo_usage_rate range)


## Task 5 — Commit *(5 min)*

```
git add -A && git commit -m "feat(lab2): linear+logistic beat baselines; overfitting exhibit"
```

**Scoreboard update** — post: logistic PR-AUC, recall@20%, best tree depth.
